In [ ]:
%gherkin
Feature: Data Standardization and Validation of delivery_dt in f_order Table

  Background:
    Given the file path is "/Workspace/stg/scripts/f_order.sql"
    And the Unity Catalog is set to "purgo_databricks"
    And the Unity Catalog Schema is "purgo_playground"
    And the table is "purgo_playground.purgo_playground.f_order"

  Scenario Outline: Validate delivery_dt format and datatype
    Given I have the column "delivery_dt" in the "f_order" table
    When I convert the "delivery_dt" into a Decimal (38,0) format
    Then the "delivery_dt" should be 100% Decimal (38,0) and in "yyyymmdd" format
    And each "delivery_dt" value should match the pattern "[0-9]{8}"
    And any invalid value should trigger an error with message "<error_message>"

    Examples:
      | error_message                               |
      | "Invalid format for delivery_dt at row N"   |
      | "Non-decimal value found in delivery_dt"    |

  Scenario: Happy path validation of delivery_dt conversion
    Given the "delivery_dt" values in source are correctly formatted as timestamps
    When converting "delivery_dt" to Decimal (38,0)
    Then the resultant "delivery_dt" should be in the format "yyyymmdd"
    And no errors are thrown

  Scenario: Error handling for incorrect "delivery_dt" entries
    Given the "delivery_dt" includes values like "202409AB"
    When validating the "delivery_dt" values
    Then record should trigger an error with message "Invalid format for delivery_dt at row N"

  Scenario Outline: Invalid range of dates
    Given "delivery_dt" values in the range "<start_date>" to "<end_date>"
    When validating range of "delivery_dt"
    Then values outside the accepted range should trigger an error "<range_error_message>"

    Examples:
      | start_date | end_date  | range_error_message                             |
      | "20000101" | "20301231"| "delivery_dt not in acceptable date range"      |

  Scenario: Impact of invalid delivery_dt on data integrity
    Given "purgo_playground.purgo_playground.f_order" has invalid "delivery_dt" entries
    When executing SQL checks for data integrity
    Then invalid entries impact subsequent processes with message "<impact_message>"

    Examples:
      | impact_message                                                      |
      | "Data integrity compromised due to invalid delivery_dt formatting"  |

  Scenario: Check integration with other tables
    Given "f_order" table is used for reports in "purgo_playground.purgo_playground.supply_chain_delivery"
    When validating consistency across related datasets
    Then ensure all integrated datasets align post-data standardization
    And flag inconsistencies with message "Data alignment issue detected in Supply Chain Delivery"
